# 02 — Subsetting

`edr-xarray` exposes four "open-time" subset filters that the EDR
server applies to every fetch issued by the resulting dataset:

| Parameter | Maps to EDR query param | Example |
|---|---|---|
| `bbox=(lon_min, lat_min, lon_max, lat_max)` | `bbox` | spatial subset |
| `datetime="..."` | `datetime` | instant or `start/end` interval |
| `z=850` or `z="1000/500"` | `z` | vertical level / range |
| `parameter_names=[...]` | filters which variables appear in the dataset | |

This notebook demonstrates each in turn against a 4D collection
(`t × z × y × x`).

The examples show how to request subsets with the public API.

In [ ]:
import httpx

server = "https://edr.example.com"  # replace with your EDR server root

collections = httpx.get(f"{server}/collections").raise_for_status().json()["collections"]
for c in collections:
    print(c["id"], "-", c.get("title", ""))

In [ ]:
collection_id = collections[0]["id"]  # or pick any id from the list above
collection_url = f"{server}/collections/{collection_id}"

## Spatial subset — `bbox`

Pass a CRS84 bounding box as `(lon_min, lat_min, lon_max, lat_max)`.
The bbox is included in *every* fetch the dataset performs, so a
spatial subset combined with a temporal subset just adds both query
parameters together.

In [ ]:
ds_spatial = xr.open_dataset(
    collection_url,
    engine="edr",
    bbox=(10.0, 40.0, 10.5, 40.5),
)
print("dims with bbox:", dict(ds_spatial.dims))
ds_spatial.close()

## Temporal subset — `datetime`

`datetime` accepts either a single ISO 8601 instant or a `start/end`
interval. Open intervals (`..`) are not supported in v1.

In [ ]:
# Single instant
ds_instant = xr.open_dataset(
    collection_url,
    engine="edr",
    datetime="2025-01-02T00:00:00Z",
)
print("instant:", dict(ds_instant.dims))
ds_instant.close()

# Closed interval
ds_interval = xr.open_dataset(
    collection_url,
    engine="edr",
    datetime="2025-01-01T00:00:00Z/2025-01-02T00:00:00Z",
)
print("interval:", dict(ds_interval.dims))
ds_interval.close()

## Vertical subset — `z`

`z` accepts a scalar (single level) or a `lo/hi` string (range).

In [ ]:
ds_level = xr.open_dataset(collection_url, engine="edr", z=850)
print("single level z=850:", dict(ds_level.dims))
ds_level.close()

ds_z_range = xr.open_dataset(collection_url, engine="edr", z="1000/500")
print("range z=1000/500:", dict(ds_z_range.dims))
ds_z_range.close()

## Parameter selection — `parameter_names`

When set, only the listed variables appear as `data_vars` on the
resulting dataset. This is a client-side filter on the metadata and is
useful when a collection advertises dozens of parameters but you only
want a handful.

In [ ]:
ds_one_var = xr.open_dataset(
    collection_url,
    engine="edr",
    parameter_names=["temperature"],
)
print("data_vars:", list(ds_one_var.data_vars))
ds_one_var.close()

## Combine

All four subsets compose freely.

In [ ]:
ds_combo = xr.open_dataset(
    collection_url,
    engine="edr",
    bbox=(10.0, 40.0, 10.5, 40.5),
    datetime="2025-01-01T00:00:00Z/2025-01-02T00:00:00Z",
    z=850,
    parameter_names=["temperature"],
)
print("combined dims:", dict(ds_combo.dims))
print("combined vars:", list(ds_combo.data_vars))
ds_combo.close()